In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "030426"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? How does this expression change with dose?

This question will be answered using the `limma` package, which was designed for analyzing gene expression data (from microarrays and RNA-seq) using linear models and empirical Bayes methods to identify differentially expressed genes or features. It is widely used in bioinformatics for handling complex experimental designs and small sample sizes ([ref](https://pmc.ncbi.nlm.nih.gov/articles/PMC4402510/)). 

9 models were tested below:
1. Control vs. NP1 0.05$\mu$g/mL
2. Control vs. NP1 0.1$\mu$g/mL
3. NP1 0.05$\mu$g/mL vs. NP1 0.1$\mu$g/mL
4. Control vs. NP1 Rhodamine B 0.05$\mu$g/mL
5. Control vs. NP1 Rhodamine B 0.1$\mu$g/mL
6. NP1 Rhodamine B 0.05$\mu$g/mL vs. NP1 Rhodamine B 0.1$\mu$g/mL
7. NP1 0.05$\mu$g/mL vs. NP1 Rhodamine B 0.05$\mu$g/mL
8. NP1 0.1$\mu$g/mL vs. NP1 Rhodamine B 0.1$\mu$g/mL
9. NP1 0.1$\mu$g/mL vs. NP2 0.1$\mu$g/mL

In [4]:
# converting to a wide format to run limma
split_protein_df = protein_df %>%
    group_by(Treatment, Dose) %>%
    group_split()

control_df = split_protein_df[[1]]
np1_0.05_df = split_protein_df[[2]]
np1_0.1_df = split_protein_df[[3]]
np1r_0.05_df = split_protein_df[[4]]
np1r_0.1_df = split_protein_df[[5]]
np2_df = split_protein_df[[6]]

wide_matrix = function(split_df, control_df){

    # recombining the control data with all other tx groups
    combined_df = rbind(split_df, control_df)
    
    wide_version = combined_df %>%
        select(Sample_ID, UniProt_ID, Conc) %>%
        pivot_wider(names_from = Sample_ID, values_from = Conc) %>%
        column_to_rownames("UniProt_ID") %>%
        as.matrix()

    return(wide_version)
}

# calling fn
wide_np1_0.05_matrix = wide_matrix(np1_0.05_df, control_df)
wide_np1_0.1_matrix = wide_matrix(np1_0.1_df, control_df)
wide_np1r_0.05_matrix = wide_matrix(np1r_0.05_df, control_df)
wide_np1r_0.1_matrix = wide_matrix(np1r_0.1_df, control_df)
wide_np2_matrix = wide_matrix(np2_df, control_df)

head(wide_np1_0.05_matrix)

,NP1_6_0.05,NP1_14_0.05,NP1_24_0.05,Control_2_NA,Control_10_NA,Control_17_NA
Q07011,1.974430,1.937668,1.983337,2.000806,1.997728,1.979591
Q15109,3.416846,3.433196,3.407561,3.394482,3.210143,3.402779
Q9UNG2,2.131393,2.139767,2.161227,2.122618,2.191947,2.093023
P31749,2.167159,2.176420,2.214149,2.183432,2.168412,2.137611
Q15389,2.491328,2.461878,2.507608,2.722051,2.675422,2.608287
P05089,3.315548,3.437836,3.386118,3.349175,3.328179,3.351498


In [5]:
# creating a sample info df
create_sample_info = function(wide_matrix, dose, Variable){

    sample_info_df = protein_df[,2:4] %>%
        unique() %>%
        filter(Treatment == 'Control' | Dose == dose & Treatment == Variable) %>%
        # ensuring the sample ids are in the same order as the previous df
        arrange(match(Sample_ID, colnames(wide_matrix)))

    return(sample_info_df)
    
    }

# calling fn
np1_0.05_sample_df = create_sample_info(wide_np1_0.05_matrix, 0.05, "NP1")
np1_0.1_sample_df = create_sample_info(wide_np1_0.1_matrix, 0.1, "NP1")
np1r_0.05_sample_df = create_sample_info(wide_np1r_0.05_matrix, 0.05, "NP1-Rhodamine B")
np1r_0.1_sample_df = create_sample_info(wide_np1r_0.1_matrix, 0.1, "NP1-Rhodamine B")
np2_sample_df = create_sample_info(wide_np2_matrix, 0.1, "NP2")

head(np1_0.05_sample_df)

,Treatment,Sample_ID,Dose
,<chr>,<chr>,<dbl>
1,NP1,NP1_6_0.05,0.05
2,NP1,NP1_14_0.05,0.05
3,NP1,NP1_24_0.05,0.05
4,Control,Control_2_NA,0.00
5,Control,Control_10_NA,0.00
6,Control,Control_17_NA,0.00


Running two group limma models.

In [6]:
get_limma_results = function(model, sample_df, wide_matrix, comparison){
    
    # running limma
    # comparing each tx group to the control
    design = model.matrix(as.formula(paste0("~", model)), data = sample_df)
    linear_fit = lmFit(wide_matrix, design)
    limma_fit = eBayes(linear_fit, robust = TRUE)
    
    treatments = colnames(design)[2:length(colnames(design))]
    limma_df = data.frame()
    for (i in 1:length(treatments)){
    
        # limma's default gives us a BH adjusted p value
        limma_results = topTable(limma_fit, coef = treatments[i], number = Inf) %>%
            rownames_to_column("UniProt_ID") %>%
            mutate(Comparison = comparison, Dose = unique(sample_df$Dose)[1]) 
    
        limma_df = rbind(limma_df, limma_results)
    
        }
    
    # cleaning up the df
    limma_df = limma_df[,c(1,2,5,6,8,9)] %>%
        # convert log fc to log2 fc
        mutate(log2FC = log2(10^logFC))
    colnames(limma_df)[3:4] = c("P Value", "P Adj")
    
    # adding in meta data
    final_df = inner_join(limma_df, unique(protein_df[,6:8]))[,c(8,1,9,5,7,3,4)]

    return(final_df)
    }

# recombining dfs
np1_np1r_0.05_sample_df = rbind(np1_0.05_sample_df[1:3,], np1r_0.05_sample_df[1:3,])
matrix_np1_npr1_0.05 = cbind(wide_np1_0.05_matrix[,1:3],wide_np1r_0.05_matrix[,1:3])
np1_np1r_0.1_sample_df = rbind(np1_0.1_sample_df[1:3,], np1r_0.1_sample_df[1:3,])
matrix_np1_npr1_0.1 = cbind(wide_np1_0.1_matrix[,1:3],wide_np1r_0.1_matrix[,1:3])
np1_np2_0.1_sample_df = rbind(np1_0.1_sample_df[1:3,], np2_sample_df[1:3,])
matrix_np1_np2 = cbind(wide_np1_0.1_matrix[,1:3],wide_np2_matrix[,1:3])
np1_0.05_0.1_sample_df = rbind(np1_0.05_sample_df[1:3,], np1_0.1_sample_df[1:3,]) 
matrix_np1_0.05_0.1 = cbind(wide_np1_0.05_matrix[,1:3], wide_np1_0.1_matrix[,1:3])
np1r_0.05_0.1_sample_df = rbind(np1r_0.05_sample_df[1:3,], np1r_0.1_sample_df[1:3,])
matrix_np1r_0.05_0.1 = cbind(wide_np1r_0.05_matrix[,1:3], wide_np1r_0.1_matrix[,1:3])

# calling fn
# np1_0.05_results_df = get_limma_results('Treatment', np1_np1r_0.05_sample_df, matrix_np1_npr1_0.05, "PET vs. PET-Rhodamine B 0.05mg/mL")
# np1_0.1_results_df = get_limma_results('Treatment', np1_np1r_0.1_sample_df, matrix_np1_npr1_0.1, "PET vs. PET-Rhodamine B 0.1mg/mL")
# np1_np2_results_df = get_limma_results('Treatment', np1_np2_0.1_sample_df, matrix_np1_np2, "PET vs. NP2 0.1mg/mL")
c_np1_0.05_results_df = get_limma_results('Treatment', np1_0.05_sample_df, wide_np1_0.05_matrix, "Control vs. PET 0.05mg/mL")
c_np1_0.1_results_df = get_limma_results('Treatment', np1_0.1_sample_df, wide_np1_0.1_matrix, "Control vs. PET 0.1mg/mL")
c_np1r_0.05_results_df = get_limma_results('Treatment', np1r_0.05_sample_df, wide_np1r_0.05_matrix, "Control vs. PET-Rhodamine B 0.05mg/mL")
c_np1r_0.1_results_df = get_limma_results('Treatment', np1r_0.1_sample_df, wide_np1r_0.1_matrix, "Control vs. PET-Rhodamine B 0.1mg/mL")
np1_0.05_0.1_results_df = get_limma_results('Dose', np1_0.05_0.1_sample_df, matrix_np1_0.05_0.1, "PET 0.05mg/mL vs. PET 0.1mg/mL")
np1r_0.05_0.1_results_df = get_limma_results('Dose', np1r_0.05_0.1_sample_df, matrix_np1r_0.05_0.1, "PET-Rhodamine B 0.05mg/mL vs.\nPET-Rhodamine B 0.1mg/mL")

Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`
Joining with `by = join_by(UniProt_ID)`


In [7]:
# combining for export
two_group_comparison_df = rbind(#np1_0.05_results_df, np1_0.1_results_df, np1_np2_results_df,
                               c_np1_0.05_results_df, c_np1_0.1_results_df, c_np1r_0.05_results_df,
                               c_np1r_0.1_results_df, np1_0.05_0.1_results_df, np1r_0.05_0.1_results_df)
head(two_group_comparison_df)

,ELISA_ID,UniProt_ID,Protein_Name,Comparison,log2FC,P Value,P Adj
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,nEL03731,P10909,CLU,Control vs. PET 0.05mg/mL,-2.7972194,4.101922e-12,1.148538e-09
2,nEL04021,O00300,TNFRSF11B,Control vs. PET 0.05mg/mL,-2.5704628,6.344761e-10,8.882666e-08
3,nEL00061,P10147,CCL3,Control vs. PET 0.05mg/mL,1.5703907,6.715775e-09,6.268057e-07
4,nEL01981,Q99988,GDF-15 (MIC-1),Control vs. PET 0.05mg/mL,1.0168714,1.130982e-05,7.916876e-04
5,nEL00441,Q9H2A7,CXCL16,Control vs. PET 0.05mg/mL,-0.8511750,1.428718e-05,8.000821e-04
6,nEL08971,Q99538,LGMN,Control vs. PET 0.05mg/mL,-0.7562914,2.988079e-05,1.394437e-03


In [8]:
# exporting
write.xlsx(two_group_comparison_df, paste0(Output,"/", "Two_Group_Results_", cur_date, ".xlsx"), 
           rowNames = FALSE)